# Lab 21 — chạy workspace cục bộ trên T4

Chọn Runtime → Change runtime type → **T4 GPU**. Chạy lần lượt các ô. Tải lên `submission/lab21_gpu_workspace.zip`; không cần commit hoặc push. Core NB1–NB5 khoảng 100–130 phút theo README. Ô 4 tạo báo cáo từ kết quả GPU. Bổ sung phản tư cá nhân trước khi nộp.

In [ ]:
# 1. Upload workspace ZIP từ máy
import io, os, pathlib, subprocess, sys, time, zipfile
from google.colab import files

uploaded = files.upload()
assert len(uploaded) == 1, "Chọn đúng một workspace ZIP."
name, payload = next(iter(uploaded.items()))
assert name.endswith(".zip"), "Cần file ZIP."
LAB_ROOT = pathlib.Path(f"/content/lab21-local-{time.time_ns()}")
LAB_ROOT.mkdir()
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    for member in archive.infolist():
        assert (LAB_ROOT / member.filename).resolve().is_relative_to(LAB_ROOT), "Đường dẫn ZIP không hợp lệ."
    archive.extractall(LAB_ROOT)
assert (LAB_ROOT / "scripts/colab_run.py").is_file(), "ZIP thiếu source lab."
os.chdir(LAB_ROOT)
print("Workspace:", LAB_ROOT)


In [ ]:
# 2. Setup GPU và kiểm tra code
import torch
assert torch.cuda.is_available(), "Chọn Runtime → Change runtime type → T4 GPU."
os.environ.update(COMPUTE_TIER="T4", MASK_MODE="assistant-only", EPOCHS="2",
                  HF_HOME=str(LAB_ROOT / ".cache/huggingface"))
for key in ("BASE_MODEL", "EVAL_LIMIT", "HF_HUB_OFFLINE", "TRANSFORMERS_OFFLINE", "ONLY", "FORCE_RETRAIN"):
    os.environ.pop(key, None)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
sys.path.insert(0, str(LAB_ROOT / "src"))
from labkit import device
print(device.banner())
subprocess.run([sys.executable, "scripts/verify.py", "--smoke"], check=True)


In [ ]:
# 3. Core: đo baseline TRƯỚC train, đầy đủ eval, cùng ngân sách step
STAGES = ["nb1", "nb2", "nb3", "nb4", "nb5"]
# Nếu run bị ngắt: trong cùng workspace, chỉ chạy các stage chưa hoàn thành.
# NB4 tự bỏ qua adapter đã lưu. Không chạy lại NB2 để sửa baseline sau khi thấy FT.
subprocess.run([sys.executable, "-u", "scripts/colab_run.py", *STAGES], check=True)


In [ ]:
# 4. Tạo báo cáo từ số đo thật và kiểm tra artefact
import json
report_args = [sys.executable, "scripts/write_gpu_report.py"]
# Nếu đã có phản tư cá nhân, thêm file này vào workspace trước khi chạy ô 4.
reflection = LAB_ROOT / "submission/PERSONAL_REFLECTION.md"
if reflection.is_file():
    report_args += ["--reflection", str(reflection)]
subprocess.run(report_args, check=True)
for artifact in ("mask_proof.json", "token_stats.json", "baselines_frozen.json", "autopsy.json", "verdict.json"):
    print("\n", artifact)
    print((LAB_ROOT / "results" / artifact).read_text(encoding="utf-8"))
print("\nruns.csv\n", (LAB_ROOT / "results/runs.csv").read_text(encoding="utf-8"))
verification = subprocess.run([sys.executable, "scripts/verify.py"], check=False)
print("verify exit:", verification.returncode)
print("Đọc REPORT.md, bổ sung phản tư cá nhân, kiểm tra ca thua và chạy verify lần cuối trước khi nộp.")


In [ ]:
# 5. Tải source + report + toàn bộ results + adapter chính về máy
import importlib.metadata
(LAB_ROOT / "submission/requirements-gpu.lock.txt").write_text(
    "\n".join(sorted(f"{dist.metadata['Name']}=={dist.version}"
                     for dist in importlib.metadata.distributions())) + "\n", encoding="utf-8")
output = pathlib.Path("/content/lab21_gpu_results.zip")
with zipfile.ZipFile(output, "w", zipfile.ZIP_DEFLATED) as archive:
    for directory in ("results", "submission", "notebooks", "src", "scripts", "data", "tests", "docs", "adapters/correct"):
        for path in sorted((LAB_ROOT / directory).rglob("*")):
            if path.is_file() and "__pycache__" not in path.parts and path.suffix != ".zip":
                archive.write(path, path.relative_to(LAB_ROOT))
    for filename in ("requirements.txt", "requirements-cpu.txt", "pyproject.toml", "README.md", "rubric.md", ".gitattributes"):
        archive.write(LAB_ROOT / filename, filename)
files.download(str(output))
